In [20]:
# Load Raw Dataset

rockyou_path = '/home/ayush_shekhar_chauhan/Desktop/Password-Strength-Analyser/ml/data/raw/rockyou.txt'

with open(rockyou_path, 'r', encoding='latin-1') as file:
    passwords = [line for line in file]


# Clean Dataset

passwords_clean = [line.rstrip('\n\r') for line in passwords]
passwords_clean = [password for password in passwords_clean if password != ""]


# Random Sampling

import random

random.seed(42)

sample_passwords = random.sample(passwords_clean, 150000)

print(len(sample_passwords))

# zxcvbn can't process data larger than 72 chars -> remove passwords more than 72 chars

sample_passwords = [password for password in sample_passwords if len(password) <= 72]

print(len(sample_passwords))


150000
149991


In [18]:
# Testing zxcvbn

from zxcvbn import zxcvbn

test_passwords = sample_passwords[:10]

for password in test_passwords :
    result = zxcvbn(password)

    print("Password:", password)
    print("Score:", result["score"])
    print("Guesses:", result["guesses"])
    print("Crack time:", result["crack_times_display"]["offline_slow_hashing_1e4_per_second"])
    print()




# Understanding -> Using Score as label isn't very much reliable we might use guesses as well

Password: Persian23
Score: 1
Guesses: 471000
Crack time: 47 seconds

Password: demetriusc
Score: 1
Guesses: 19922
Crack time: 2 seconds

Password: franciska
Score: 1
Guesses: 34600
Crack time: 3 seconds

Password: 411639213
Score: 2
Guesses: 27604000
Crack time: 46 minutes

Password: plopper11
Score: 3
Guesses: 156463000
Crack time: 4 hours

Password: ry2048
Score: 1
Guesses: 110000
Crack time: 11 seconds

Password: singh129
Score: 2
Guesses: 2210000
Crack time: 4 minutes

Password: 19dec28
Score: 2
Guesses: 10000001
Crack time: 17 minutes

Password: 4569882
Score: 1
Guesses: 740000
Crack time: 1 minute

Password: ilovederk
Score: 3
Guesses: 100830000
Crack time: 3 hours



In [22]:
# Converting Raw Data to pandas DataFrame 

import pandas as pd

temp_df = pd.DataFrame(sample_passwords, columns=["passwords"])

# Function To get SCORE and GUESSES

def get_zxcvbn_data (password) :
    result = zxcvbn(password)

    return result['score'], result['guesses']

zxcvbn_results = temp_df['passwords'].apply(get_zxcvbn_data)

zxcvbn_results = zxcvbn_results.apply(pd.Series)
zxcvbn_results.columns = ["score", "guesses"]

# Combining DataFrames 1 -> temp_df with Sample_passwords and 2 -> zxcvbn Results

df = pd.concat([temp_df, zxcvbn_results], axis=1)

print(df)

           passwords  score                        guesses
0          Persian23      1                         471000
1         demetriusc      1                          19922
2          franciska      1                          34600
3          411639213      2                       27604000
4          plopper11      3                      156463000
...              ...    ...                            ...
149986      lover802      1                         722000
149987  ???FUTURO158      4  10527668159.99999991429376678
149988    0914325910      3                      146010000
149989         fa333      1                          20000
149990       rover11      1                         188640

[149991 rows x 3 columns]


In [29]:
print(df['score'].value_counts().sort_index())

print(df['score'].value_counts(normalize=True).sort_index() * 100)

score
0      329
1    53917
2    54574
3    31350
4     9821
Name: count, dtype: int64
score
0     0.219346
1    35.946823
2    36.384850
3    20.901254
4     6.547726
Name: proportion, dtype: float64
